# Day 6 · Morning exercises (Exercise A)

**09:45–11:10 · then everyone back in the room at 11:10 for Peer review A**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. Keep the folders `data/` and `models/` next to this notebook. The afternoon continues in `day-6-exercises-afternoon.ipynb`.

This morning: one person's hits and misses, modelled with a logistic regression in Stan.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| A1 | Hit or miss: one person's 60 attempts; probability, odds and log-odds | 15 min |
| A2 | Priors on the log-odds scale | 15 min |
| A3 | Fit the logistic regression with a design matrix; odds ratios | 20 min |
| A4 | Predictions for chosen cases and a grouped check | 20 min |
| — | Exit response and your group's question set | 15 min |

**Roles this block:** coder, checker, two question writers, interviewer, reporter. Rotate them in the afternoon.

**The data.** The class's own Day 1 time games (`data/timing-class.csv`): 56 people, each with ten attempts at 5, 10 and 15 seconds **without feedback**, then the same **with feedback**. The data are as recorded, including a few unusual attempts (a click too early, for example). A **hit** is an attempt within 0.5 s of the target. This morning we model **one person**: yourself, or G1P01.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
theme_set(theme_minimal(base_size = 14) + theme(legend.position = "top"))
DATA <- "data"; MODELS <- "models"     # folders next to this notebook
condition_colours <- c("without feedback" = "navy", "with feedback" = "tomato")

In [ ]:
# The class's Day 1 data: both games, 56 people, 30 attempts per game
d_all <- read_csv(file.path(DATA, "timing-class.csv"), show_col_types = FALSE) |>
  mutate(hit       = as.integer(abs(elapsed_seconds - target_seconds) <= 0.5),   # 1 = within 0.5 s of the target
         feedback  = as.integer(feedback_condition == "feedback"),              # 1 = with feedback
         x         = target_seconds - 10,                                       # centred target
         condition = factor(if_else(feedback == 1, "with feedback", "without feedback"),
                            levels = c("without feedback", "with feedback")))
nrow(d_all)

me <- "G1P01"                       # or your own ID from Day 1, for example "G3P08"
d <- d_all |> filter(participant == me)
nrow(d)

## A1 (15 min): hit or miss

**The model** (*slides "Hit or miss?", "A yes/no outcome", "Probability, odds and log-odds" and "A second predictor: feedback"*). Attempt i is a hit (yᵢ = 1) or a miss (yᵢ = 0), with xᵢ = target − 10 and fᵢ = 1 with feedback:

yᵢ ~ Bernoulli(pᵢ),   logit(pᵢ) = α + β₁ · xᵢ + β₂ · fᵢ.

**Tasks.**

1. Make a table of hits out of attempts for each target and condition, and plot the share of hits against the target, one colour per condition.
2. Her share of hits at 10 s without feedback is p. Compute its odds p / (1 − p) and log-odds `qlogis(p)`. What would a share of 0 or 1 give, and why is that a problem for a raw estimate?
3. In one sentence each: what do α, β₁ and β₂ mean for this person?

In [ ]:
# Task 1
# cells <- d |> group_by(target_seconds, condition) |>
#   summarise(hits = sum(hit), attempts = n(), .groups = "drop") |> mutate(share = hits / attempts)
# ggplot(cells, aes(x = target_seconds, y = share, colour = condition)) + geom_line() + geom_point(size = 4) +
#   scale_colour_manual(values = condition_colours, name = NULL) + labs(x = "target (s)", y = "share of hits")

# Task 2
# p <- cells |> filter(target_seconds == 10, condition == "without feedback") |> pull(share)
# c(p = p, odds = p / (1 - p), log_odds = qlogis(p))

**Your answer:** write a few sentences here.

## A2 (15 min): priors on the log-odds scale

**The priors** (*slides "Priors on the log-odds scale", "Priors for this model" and "Prior predictive: curves of P(hit)"*): α ~ Normal(0, 1.5), β₁ ~ Normal(0, 0.5) per second, β₂ ~ Normal(0, 1).

**Tasks.**

1. Draw 4000 values of α from Normal(0, 1.5) and from Normal(0, 10). For each, plot the implied P(hit) = `plogis(alpha)` at 10 s without feedback, and give the share of draws below 0.01 or above 0.99.
2. What range of odds ratios for feedback does β₂ ~ Normal(0, 1) allow (±2 prior SDs)? Is that reasonable for the effect of feedback on hitting a target?
3. Simulate 50 prior curves of P(hit) against the target, with and without feedback, from the three priors. Do they look like possible hit rates?

In [ ]:
# Task 1
S <- 4000
# prior_p <- bind_rows(tibble(prior = "alpha ~ Normal(0, 1.5)", p = plogis(rnorm(S, 0, 1.5))),
#                      tibble(prior = "alpha ~ Normal(0, 10)",  p = plogis(rnorm(S, 0, 10))))
# prior_p |> group_by(prior) |> summarise(share_extreme = mean(p < 0.01 | p > 0.99))

# Task 2: exp(c(-2, 2))

# Task 3
# prior_curves <- tibble(draw = 1:50, alpha = rnorm(50, 0, 1.5), b1 = rnorm(50, 0, 0.5), b2 = rnorm(50, 0, 1)) |>
#   cross_join(expand_grid(feedback = c(0, 1), target = seq(3, 17, by = 0.5))) |>
#   mutate(p = plogis(alpha + b1 * (target - 10) + b2 * feedback), ...)

**Your answer:** write a few sentences here.

## A3 (20 min): fit the logistic regression with a design matrix

**The Stan program** (*slides "The design matrix", "The logistic model in Stan" and "The fit"*): `models/logistic.stan`. X has one row per attempt and one column per predictor; `prior_sd` gives each column its prior SD; `X_new` holds cases to predict (used in A4).

**Tasks.**

1. Build X with the columns x and feedback, and X_new with the six cases (targets 5, 10, 15 s, without and with feedback). Fit the model (4 chains, `seed = 2026`) and check the diagnostics.
2. Report α, β₁ and β₂ with 90% intervals, and the odds ratios exp(β₁) per second, exp(5 β₁) per 5 s and exp(β₂) for feedback, with 90% intervals.
3. Compute P(β₂ > 0 | y). What can you say about feedback for this person, and why is the interval so wide?

In [ ]:
# Task 1
# X <- cbind(x = d$x, feedback = d$feedback)
# cases <- expand_grid(feedback = c(0, 1), target_seconds = c(5, 10, 15))
# X_new <- cbind(x = cases$target_seconds - 10, feedback = cases$feedback)
# stan_data <- list(N = nrow(d), K = ncol(X), X = X, y = d$hit, prior_sd = c(0.5, 1), M = nrow(X_new), X_new = X_new)
# mod <- cmdstan_model(file.path(MODELS, "logistic.stan"))
# fit <- mod$sample(data = stan_data, chains = 4, seed = 2026, refresh = 0)

# Task 2 and 3
# draws <- fit$draws(format = "df") |> as_tibble()
# quantile(exp(draws$`beta[2]`), c(0.05, 0.95));  mean(draws$`beta[2]` > 0)

**Your answer:** write a few sentences here.

## A4 (20 min): predictions for chosen cases and a grouped check

**The idea** (*slides "Predicted probabilities for chosen cases" and "A predictive check for yes/no data"*). `p_new[k]` is P(hit) for row k of `X_new`. A single 0 or 1 cannot look unusual, so the check uses grouped counts: hits out of 10 for each target and condition.

**Tasks.**

1. Give P(hit) with 90% intervals for the six cases, and plot them.
2. Her next 10 attempts at 15 s with feedback (case 6): simulate the number of hits, give its 90% interval and P(5 or more hits).
3. Grouped check: replicate her 60 attempts once per draw, count the hits in each target and condition, and compare the 90% range of replicated counts with her observed counts. Any misfit?

In [ ]:
# Task 1
# predictions <- cases |> mutate(mean = fit$summary("p_new")$mean,
#                                lo = fit$summary("p_new", ~quantile(.x, 0.05))[[2]],
#                                hi = fit$summary("p_new", ~quantile(.x, 0.95))[[2]])

# Task 2
# y_next <- rbinom(nrow(draws), 10, draws$`p_new[6]`)

# Task 3
# replicated <- draws |>
#   select(.draw, alpha, b1 = `beta[1]`, b2 = `beta[2]`) |>
#   cross_join(d |> select(target_seconds, condition, x, feedback)) |>
#   mutate(y_rep = rbinom(n(), 1, plogis(alpha + b1 * x + b2 * feedback))) |>
#   group_by(.draw, target_seconds, condition) |>
#   summarise(rep_hits = sum(y_rep), .groups = "drop")

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: How would you explain β₂ and exp(β₂) to a colleague who has never seen a logistic regression? Why is a predicted probability for a chosen case often a clearer way to report the result?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 11:10 for Peer review A.** Save this notebook (Ctrl/Cmd + S). After lunch, open `day-6-exercises-afternoon.ipynb`.